# 6.1 基线模型

先用三种不需要训练的笨办法预测：直接拿上月值、近 3 月均值、近 6 月均值。
它们在验证期的误差就是模型必须超过的那条线——模型要是还不如"拿上个月的数"，就没有价值。

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

import dsflow

ROOT = Path.cwd()
while not (ROOT / "dsflow.yaml").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from demo_lib import mae, out

OUT = out("6.1")
FEAT = out("4.1") / "features.parquet"
METHODS = {"上月值": "lag1", "近3月均值": "mean3", "近6月均值": "mean6"}
run = dsflow.start_run("6.1", project=ROOT, hypothesis="三种不用训练的方法给出模型必须超过的误差")

run.log_input(FEAT, name="features")
valid = pd.read_parquet(FEAT).query("划分 == '验证'")
metrics = {f"MAE_{k}": mae(valid["需求量"], valid[col]) for k, col in METHODS.items()}
print(f"验证期 {len(valid):,} 个 SKU 月")
print(json.dumps(metrics, ensure_ascii=False, indent=1))


验证期 9,000 个 SKU 月
{
 "MAE_上月值": 8.7692,
 "MAE_近3月均值": 7.4249,
 "MAE_近6月均值": 6.9544
}


In [2]:
by_month = {}
for month, d in valid.groupby("月份"):
    by_month[month] = {f"MAE_{k}": mae(d["需求量"], d[col]) for k, col in METHODS.items()}
    run.log_metrics(by_month[month], fold=month)
pd.DataFrame(by_month).T


,MAE_上月值,MAE_近3月均值,MAE_近6月均值
2026-01,8.9777,7.4683,6.9283
2026-02,8.7200,7.3150,6.9061
2026-03,8.6100,7.4912,7.0287


In [3]:
best = min(metrics, key=metrics.get)
run.log_metrics(metrics)
(OUT / "baseline_metrics.json").write_text(
    json.dumps({"验证期": metrics, "逐月": by_month, "最好的基线": best}, ensure_ascii=False, indent=1), encoding="utf-8")
run.log_artifact(OUT / "baseline_metrics.json", purpose="三种基线在验证期的 MAE（含逐月）", kind="table")
conclusion = f"验证期 {len(valid):,} 个 SKU 月：最好的基线是{best.removeprefix('MAE_')}，MAE {metrics[best]}"
run.set_conclusion(conclusion, validity="有效")
run.end()
print(conclusion)


验证期 9,000 个 SKU 月：最好的基线是近6月均值，MAE 6.9544
